# Stage 4 sparse spatial-loss ablation

This Colab is a controlled Stage 4 experiment asking whether raw dense voxel MSE causally encourages low-amplitude, background-dominated predictions for sparse PubMed and Nilearn maps.

It does **not** change the production loss. Every run retains:

- the released branch-specific Stage 1 AE, fully frozen and in evaluation mode;
- the published empty-string-centered, unit-normalized SPECTER2 cache;
- the published train/validation/test splits;
- a fresh `768 → 512 → ReLU → 384` projector;
- the raw 384-D latent convention and raw decoder output;
- the retained AdamW settings, batch strategy, validation cadence, epoch limit, gradient clipping, and early stopping.

The test split is evaluated only after checkpoint selection. The experiment is intentionally separate from `neurovlm.training.text_to_brain`.


## 1. Mount Drive and check out the repository

For an archival run, replace `REPO_REF` with an immutable commit SHA and optionally set `EXPECTED_COMMIT`. A branch name is convenient during notebook development but is weaker provenance.


In [ ]:
from pathlib import Path
import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/neurovlm/neurovlm.git"
REPO_REF = "neurovlm_experiments"
EXPECTED_COMMIT = None  # strongly recommended for the final archival run
REPO_DIR = Path("/content/neurovlm" if IN_COLAB else Path.cwd()).resolve()

if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--all", "--tags", "--prune"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
if subprocess.run(
    ["git", "show-ref", "--verify", "--quiet", f"refs/remotes/origin/{REPO_REF}"],
    cwd=REPO_DIR,
).returncode == 0:
    subprocess.run(["git", "merge", "--ff-only", f"origin/{REPO_REF}"], cwd=REPO_DIR, check=True)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
if EXPECTED_COMMIT is not None and RESOLVED_COMMIT != EXPECTED_COMMIT:
    raise RuntimeError(f"Expected commit {EXPECTED_COMMIT}, resolved {RESOLVED_COMMIT}")
print("Repository:", REPO_DIR)
print("Configured ref:", REPO_REF)
print("Resolved commit:", RESOLVED_COMMIT)


## 2. Install dependencies

The editable install ensures this notebook exercises the checked-out code. Restarting the runtime is not required.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{REPO_DIR}[metrics,viz,notebook]",
    ],
    check=True,
)
src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ["PYTHONPATH"] = src_path + os.pathsep + os.environ.get("PYTHONPATH", "")
os.chdir(REPO_DIR)


## 3. Experiment configuration

`FAST_ABLATION=True` keeps every selected branch and every loss configuration, but limits epochs and train/evaluation batches. It is a pipeline smoke test, not evidence for the final report.

The foreground sweep creates four independent runs. Optional automatic balancing is disabled by default so the declared coefficients remain literal. When enabled, a fixed training batch initializes and then freezes multipliers; requested relative weights are clipped to the configured gradient-ratio band.


In [ ]:
BRANCHES_TO_RUN = [
    "mixed_to_pubmed", "pubmed",
    "mixed_to_nilearn", "nilearn",
    "mixed_to_neurovault", "neurovault",
]
FAST_ABLATION = False

SEED = 42
PROJECTOR_SEED = 42
EPOCHS = 100
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 0
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 0.0
VALIDATE_EVERY_EPOCHS = 1
SCHEDULER = "none"

AMP = True
AMP_DTYPE = "auto"  # BF16 on supported A100/H100, otherwise FP16
LAMBDA_FG_VALUES = [1.0, 4.0, 10.0, 25.0]
LAMBDA_POSITIVE = 4.0
LAMBDA_DICE = 0.25
LAMBDA_CORR = 0.10
HYBRID_WEIGHTS = {"dense": 0.25, "foreground": 1.0, "dice": 0.25, "correlation": 0.10}
SOFTPLUS_BETA = 5.0
SOFT_TOP5_TEMPERATURE = 0.10
EPSILON = 1e-6

AUTO_GRADIENT_BALANCE = False
GRADIENT_BALANCE_FACTOR = 3.0
GRADIENT_BALANCE_MIN_WEIGHT = 1e-4
GRADIENT_BALANCE_MAX_WEIGHT = 1e4

RUN_UNIT_TESTS = True
RUN_TINY_CONTROLS = True
RUN_FULL_TRAINING = True
TINY_N = 32
TINY_STEPS = 750
TINY_LR = 1e-3
SEMANTIC_MAX_EXAMPLES = 1024
SEMANTIC_NEIGHBORS = 10
DISTRIBUTION_SAMPLE_VOXELS = 250_000
EXAMPLES_PER_RUN = 6
FULL_DATA_LIMIT = None

DRIVE_OUTPUT_BASE = Path(
    "/content/drive/MyDrive/neurovlm/stage4_sparse_spatial_loss_ablation"
    if IN_COLAB else REPO_DIR / "runs" / "stage4_sparse_spatial_loss_ablation"
)
RESUME_EXPERIMENT_DIR = None
AUTO_RESUME_ACTIVE = True

if FAST_ABLATION:
    EPOCHS = 2
    EARLY_STOPPING_PATIENCE = None
    TINY_STEPS = 75
    SEMANTIC_MAX_EXAMPLES = 64
    FULL_DATA_LIMIT = 128
    MAX_TRAIN_BATCHES = 2
    MAX_EVAL_BATCHES = 2
else:
    MAX_TRAIN_BATCHES = None
    MAX_EVAL_BATCHES = None

assert SCHEDULER == "none"
assert VALIDATE_EVERY_EPOCHS == 1


## 4. Environment, determinism, branches, and provenance

The two branch types per domain are the existing Stage 4 controls: `mixed_to_*` uses released Stage 1A mixed-AE resources; the domain-only branch uses the released Stage 1B domain-finetuned AE. Pairing, split, AE, and cache identities are bound into every checkpoint.


In [ ]:
import copy, csv, importlib.metadata, json, math, platform, random, tempfile
from dataclasses import dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from matplotlib import pyplot as plt
from torch import Tensor, nn
from torch.utils.data import DataLoader

from neurovlm import retrieval_resources as rr
from neurovlm.atlas_free_dataset import AtlasFreeCNNDataProvider
from neurovlm.atlas_free_text import AtlasFreeContrastiveCollator, AtlasFreeTextEmbeddingLookup
from neurovlm.cnn import CNNTextToBrainModel, GenerativeTextToAELatent, autoencoder_from_payload
from neurovlm.evaluation.spatial import reconstruction_metrics
from neurovlm.evaluation.text_to_brain_audit import (
    ae_ceiling_bypass, audit_pairings, audit_raw_latent_path,
    audit_text_preprocessing, autoencoder_identity, frozen_ae_determinism,
    tiny_overfit_projector,
)
from neurovlm.experiments.stage4_latent_ablation import (
    AblationCheckpointManager, encode_stage1_latents, resolve_amp_dtype,
    split_fingerprint, text_cache_identity,
)
from neurovlm.pipelines import (
    atomic_write_csv, atomic_write_json, environment_provenance,
    git_provenance, sha256_file, sha256_state_dict, sha256_value,
)
from neurovlm.semantic_evaluation import evaluate_semantic_neighbor_retrieval
from neurovlm.training.text_to_brain import (
    _autoencoder_state_provenance, _text_cache_provenance,
    _validate_recorded_autoencoder_state, _validate_recorded_text_cache,
)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MIXED_PRECISION_DTYPE = resolve_amp_dtype(DEVICE, AMP_DTYPE)
packages = [
    "neurovlm", "torch", "numpy", "pandas", "matplotlib",
    "nilearn", "nibabel", "huggingface-hub", "transformers",
]
versions = {}
for package in packages:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None
ENVIRONMENT = {
    **environment_provenance(packages),
    "python_full": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
    "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    "training_amp_dtype": str(MIXED_PRECISION_DTYPE),
    "git": git_provenance(REPO_DIR),
    "configured_ref": REPO_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "packages": versions,
}
print(json.dumps(ENVIRONMENT, indent=2, default=str))


In [ ]:
BRANCH_SPECS = {
    "mixed_to_pubmed": {"domain": "pubmed", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "pubmed": {"domain": "pubmed", "variant": "finetuned", "stage1": "1B", "ae_variant": "pubmed"},
    "mixed_to_nilearn": {"domain": "nilearn", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "nilearn": {"domain": "nilearn", "variant": "finetuned", "stage1": "1B", "ae_variant": "nilearn"},
    "mixed_to_neurovault": {"domain": "neurovault", "variant": "mixed_baseline", "stage1": "1A", "ae_variant": "mixed"},
    "neurovault": {"domain": "neurovault", "variant": "finetuned", "stage1": "1B", "ae_variant": "neurovault"},
}
unknown = sorted(set(BRANCHES_TO_RUN) - set(BRANCH_SPECS))
if unknown:
    raise ValueError(f"Unknown branches: {unknown}")
for name, spec in BRANCH_SPECS.items():
    spec["branch"] = name

def utc_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def resolve_experiment_root():
    DRIVE_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    pointer = DRIVE_OUTPUT_BASE / "ACTIVE_EXPERIMENT.json"
    if RESUME_EXPERIMENT_DIR is not None:
        root = Path(RESUME_EXPERIMENT_DIR)
    elif AUTO_RESUME_ACTIVE and pointer.exists():
        active = json.loads(pointer.read_text())
        candidate = Path(active["path"])
        root = candidate if active.get("state") != "completed" and candidate.exists() else DRIVE_OUTPUT_BASE / utc_stamp()
    else:
        root = DRIVE_OUTPUT_BASE / utc_stamp()
    root.mkdir(parents=True, exist_ok=True)
    atomic_write_json(pointer, {"path": str(root), "state": "running", "updated_at": utc_stamp()})
    return root, pointer

def freeze_module(module):
    module.eval()
    for parameter in module.parameters():
        parameter.requires_grad_(False)
    return module

def make_loader(dataset, lookup, *, batch_size, shuffle, seed):
    lookup.validate_dataset(dataset.rows)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS,
        collate_fn=AtlasFreeContrastiveCollator(lookup, (36, 45, 38)),
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        generator=torch.Generator().manual_seed(seed),
    )

def fixed_batch(dataset, lookup, n=BATCH_SIZE):
    return next(iter(make_loader(dataset, lookup, batch_size=n, shuffle=False, seed=SEED)))

def load_branch_resources(branch_name):
    spec = BRANCH_SPECS[branch_name]
    filename = rr.CNN_AUTOENCODER_FILENAMES[spec["ae_variant"]]
    ae_path = Path(rr._download_from_hf(rr.ATLAS_FREE_CNN_MODEL_REPO_ID, filename, repo_type="model"))
    payload = torch.load(ae_path, map_location="cpu", weights_only=True)
    autoencoder = freeze_module(autoencoder_from_payload(payload))
    autoencoder.encoder.eval()
    autoencoder.decoder.eval()
    provider = AtlasFreeCNNDataProvider(domain=spec["domain"], limit=FULL_DATA_LIMIT)
    semantic_model = freeze_module(rr._load_cnn_contrastive(branch_name))
    return spec, ae_path, autoencoder, provider, semantic_model

def build_provenance(spec, ae_path, autoencoder, provider, lookup, audit_dir):
    ae_source = _autoencoder_state_provenance(
        {
            "kind": "released", "path": str(ae_path.resolve()),
            "sha256": sha256_file(ae_path), "branch": spec["branch"],
            "domain": spec["domain"], "stage1": spec["stage1"],
            "variant": spec["variant"], "loader_variant": spec["ae_variant"],
        },
        autoencoder,
    )
    cache_source = _text_cache_provenance(lookup)
    _validate_recorded_autoencoder_state(ae_source, autoencoder)
    _validate_recorded_text_cache(cache_source, _text_cache_provenance(lookup))
    text_audit = audit_text_preprocessing(lookup)
    if not text_audit["passed"]:
        raise RuntimeError(f"Strict text-cache audit failed: {text_audit}")
    pairings = {}
    for split in ("train", "val", "test"):
        dataset = getattr(provider, split)
        pairings[split] = audit_pairings(
            dataset, lookup, minimum=min(100, len(dataset)), output_dir=audit_dir
        )
        if not pairings[split]["passed"]:
            raise RuntimeError(f"{split} pairing audit failed")
    model_probe = CNNTextToBrainModel(GenerativeTextToAELatent(), autoencoder)
    raw_path_audit = audit_raw_latent_path(model_probe)
    if not raw_path_audit["passed"]:
        raise RuntimeError(f"Raw-latent path audit failed: {raw_path_audit}")
    return {
        "autoencoder": ae_source,
        "autoencoder_identity": autoencoder_identity(
            autoencoder, checkpoint=ae_path, domain=spec["domain"], branch=spec["branch"]
        ),
        "text_cache": {**cache_source, **text_cache_identity(lookup)},
        "text_preprocessing_audit": text_audit,
        "pairing_audits": pairings,
        "splits": {split: split_fingerprint(getattr(provider, split)) for split in ("train", "val", "test")},
        "branch": dict(spec),
        "projector": {"name": "GenerativeTextToAELatent", "layers": [768, 512, "ReLU", 384]},
        "latent_convention": "raw_384d_stage1_ae_latent",
        "decoder_output": "raw_unclamped_for_training",
        "git_commit": RESOLVED_COMMIT,
    }

EXPERIMENT_ROOT, ACTIVE_POINTER = resolve_experiment_root()


## 5. Differentiable spatial losses

All reductions are per-example before batch averaging so maps contribute equally.

- **Dense MSE:** raw prediction and target; no clamping.
- **Foreground-weighted MSE:** squared error weighted by positive target intensity, normalized by the total intensity weight. This differs from the uniform positive mask.
- **Positive-only MSE:** uniform mean over `target > 0`; an all-background example safely falls back to dense MSE.
- **Balanced foreground/background:** the arithmetic mean of separately normalized positive and non-positive MSE.
- **Soft top-5 Dice:** predictions pass through `softplus(beta=5)`. A sigmoid around the detached per-example 95th percentile produces soft predicted membership; the target top-5% mask is hard because the target is constant. The temperature is scaled by the prediction standard deviation. This component never clamps the raw decoder output.
- **Correlation:** `1 - Pearson r` on raw, centered prediction/target voxels in FP32. Constant examples contribute zero correlation rather than NaN.

Every component uses epsilon protection and is computed in FP32 even inside autocast.


In [ ]:
COMPONENT_NAMES = ("latent", "dense", "foreground", "positive", "balanced", "dice", "correlation")
SPATIAL_COMPONENTS = COMPONENT_NAMES[1:]

@dataclass
class LossResult:
    total: Tensor
    components: dict[str, Tensor]
    weighted: dict[str, Tensor]
    prediction_latent: Tensor
    prediction_volume: Tensor

def _flat_fp32(value):
    value = torch.nan_to_num(value.float(), nan=0.0, posinf=1e4, neginf=-1e4)
    return value.reshape(value.shape[0], -1)

def _masked_per_example_mean(values, mask, fallback):
    mask = mask.to(values.dtype)
    count = mask.sum(dim=1)
    selected = (values * mask).sum(dim=1) / count.clamp_min(1.0)
    return torch.where(count > 0, selected, fallback)

def soft_top5_dice_loss(prediction, target, *, temperature=SOFT_TOP5_TEMPERATURE, epsilon=EPSILON):
    pred = F.softplus(_flat_fp32(prediction), beta=SOFTPLUS_BETA)
    truth = _flat_fp32(target).clamp_min(0)
    pred_threshold = torch.quantile(pred.detach(), 0.95, dim=1, keepdim=True)
    scale = pred.detach().std(dim=1, keepdim=True, unbiased=False).clamp_min(epsilon)
    pred_support = torch.sigmoid((pred - pred_threshold) / (temperature * scale + epsilon))
    k = max(1, int(math.ceil(truth.shape[1] * 0.05)))
    target_indices = truth.topk(k, dim=1, largest=True, sorted=False).indices
    target_support = torch.zeros_like(truth).scatter(1, target_indices, 1.0)
    intersection = (pred_support * target_support).sum(dim=1)
    denominator = pred_support.sum(dim=1) + target_support.sum(dim=1)
    dice = (2.0 * intersection + epsilon) / (denominator + epsilon)
    return (1.0 - dice).mean()

def raw_spatial_correlation_loss(prediction, target, *, epsilon=EPSILON):
    pred = _flat_fp32(prediction)
    truth = _flat_fp32(target)
    pred = pred - pred.mean(dim=1, keepdim=True)
    truth = truth - truth.mean(dim=1, keepdim=True)
    numerator = (pred * truth).sum(dim=1)
    denominator = pred.square().sum(dim=1).sqrt() * truth.square().sum(dim=1).sqrt()
    correlation = torch.where(
        denominator > epsilon, numerator / denominator.clamp_min(epsilon),
        torch.zeros_like(denominator),
    ).clamp(-1.0, 1.0)
    return (1.0 - correlation).mean()

def spatial_loss_components(prediction, target):
    pred = _flat_fp32(prediction)
    truth = _flat_fp32(target)
    squared = (pred - truth).square()
    dense_by_example = squared.mean(dim=1)
    positive_mask = truth > 0
    background_mask = ~positive_mask
    positive = _masked_per_example_mean(squared, positive_mask, dense_by_example).mean()
    background = _masked_per_example_mean(squared, background_mask, dense_by_example).mean()
    balanced = 0.5 * (positive + background)
    intensity = truth.clamp_min(0)
    intensity = intensity / intensity.amax(dim=1, keepdim=True).clamp_min(EPSILON)
    intensity_sum = intensity.sum(dim=1)
    intensity_weighted = (squared * intensity).sum(dim=1) / intensity_sum.clamp_min(EPSILON)
    foreground = torch.where(
        intensity_sum > EPSILON, intensity_weighted, dense_by_example
    ).mean()
    return {
        "dense": dense_by_example.mean(),
        "foreground": foreground,
        "positive": positive,
        "balanced": balanced,
        "dice": soft_top5_dice_loss(prediction, target),
        "correlation": raw_spatial_correlation_loss(prediction, target),
    }

def loss_specs():
    specs = [
        {"run_name": "baseline_dense_mse", "objective": "baseline_dense_mse", "weights": {"dense": 1.0}},
    ]
    specs.extend(
        {
            "run_name": f"foreground_weighted_mse_lambda{int(value)}",
            "objective": "foreground_weighted_mse", "lambda_fg": value,
            "weights": {"dense": 1.0, "foreground": value},
        }
        for value in LAMBDA_FG_VALUES
    )
    specs.extend([
        {
            "run_name": f"positive_voxel_mse_lambda{LAMBDA_POSITIVE:g}".replace(".", "p"),
            "objective": "positive_voxel_mse", "lambda_positive": LAMBDA_POSITIVE,
            "weights": {"dense": 1.0, "positive": LAMBDA_POSITIVE},
        },
        {
            "run_name": "balanced_foreground_background",
            "objective": "balanced_foreground_background", "weights": {"balanced": 1.0},
        },
        {
            "run_name": "soft_top5_dice",
            "objective": "soft_top5_dice", "lambda_dice": LAMBDA_DICE,
            "weights": {"dense": 1.0, "dice": LAMBDA_DICE},
        },
        {
            "run_name": "correlation_loss",
            "objective": "correlation_loss", "lambda_corr": LAMBDA_CORR,
            "weights": {"dense": 1.0, "correlation": LAMBDA_CORR},
        },
        {
            "run_name": "hybrid_sparse", "objective": "hybrid_sparse",
            "weights": dict(HYBRID_WEIGHTS),
        },
    ])
    return specs

LOSS_SPECS = loss_specs()

def effective_weights(spec, calibration):
    weights = {"latent": 1.0, **{name: 0.0 for name in SPATIAL_COMPONENTS}}
    weights.update({name: float(value) for name, value in spec["weights"].items()})
    if AUTO_GRADIENT_BALANCE:
        for name in SPATIAL_COMPONENTS:
            base = weights[name]
            if base > 0:
                relative = min(GRADIENT_BALANCE_FACTOR, max(1 / GRADIENT_BALANCE_FACTOR, base))
                weights[name] = relative * float(calibration["recommended_weights"][name])
    return weights

def compute_loss(projector, autoencoder, text, target_volume, target_latent, weights):
    prediction_latent = projector(text)
    prediction_volume = autoencoder.decoder(prediction_latent)
    components = {
        "latent": F.mse_loss(prediction_latent.float(), target_latent.detach().float()),
        **spatial_loss_components(prediction_volume, target_volume),
    }
    weighted = {name: components[name] * float(weights.get(name, 0.0)) for name in COMPONENT_NAMES}
    total = sum(weighted.values())
    return LossResult(total, components, weighted, prediction_latent, prediction_volume)

assert len(LOSS_SPECS) == 10
pd.DataFrame(LOSS_SPECS)


## 6. Gradient calibration and finite-gradient tests

Calibration uses the first fixed, unshuffled **training** batch for each branch. It measures the same fresh seeded projector separately for every component. The recommended coefficient is `latent_gradient_norm / component_gradient_norm`, clipped only for numerical safety. When automatic balancing is enabled, coefficients are frozen before training and recorded; no test example participates.


In [ ]:
def parameter_gradient_norm(loss, parameters, *, retain_graph):
    gradients = torch.autograd.grad(
        loss, parameters, retain_graph=retain_graph, allow_unused=True
    )
    total = sum(
        gradient.detach().float().square().sum()
        for gradient in gradients if gradient is not None
    )
    return float(total.sqrt())

def gradient_calibration(autoencoder, batch):
    seed_everything(PROJECTOR_SEED)
    projector = GenerativeTextToAELatent(768, 512, 384).to(DEVICE).train()
    autoencoder = autoencoder.to(DEVICE).eval()
    text = batch["text_embedding"].to(DEVICE)
    target = batch["volume"].to(DEVICE)
    with torch.no_grad():
        target_latent = autoencoder.encoder(target)
    with torch.autocast(
        device_type=DEVICE.type, dtype=MIXED_PRECISION_DTYPE,
        enabled=AMP and DEVICE.type == "cuda" and MIXED_PRECISION_DTYPE != torch.float32,
    ):
        result = compute_loss(
            projector, autoencoder, text, target, target_latent,
            {name: 1.0 for name in COMPONENT_NAMES},
        )
    parameters = tuple(projector.parameters())
    norms = {}
    for index, name in enumerate(COMPONENT_NAMES):
        norms[name] = parameter_gradient_norm(
            result.components[name], parameters,
            retain_graph=index < len(COMPONENT_NAMES) - 1,
        )
    latent_norm = norms["latent"]
    recommended = {"latent": 1.0}
    for name in SPATIAL_COMPONENTS:
        value = latent_norm / max(norms[name], EPSILON)
        recommended[name] = min(GRADIENT_BALANCE_MAX_WEIGHT, max(GRADIENT_BALANCE_MIN_WEIGHT, value))
    return {
        "batch_map_ids": [str(value) for value in batch["map_id"]],
        "batch_text_ids": [str(value) for value in batch["text_id"]],
        "batch_size": len(target),
        "source_split": "train",
        "projector_seed": PROJECTOR_SEED,
        "amp_dtype": str(MIXED_PRECISION_DTYPE),
        "component_gradient_norms": norms,
        "recommended_weights": recommended,
        "auto_balance_enabled": AUTO_GRADIENT_BALANCE,
        "target_factor_band": [1 / GRADIENT_BALANCE_FACTOR, GRADIENT_BALANCE_FACTOR],
    }

def amp_component_tests(autoencoder, batch, dtypes):
    rows = []
    for dtype in dtypes:
        for component in COMPONENT_NAMES:
            seed_everything(PROJECTOR_SEED)
            projector = GenerativeTextToAELatent().to(DEVICE).train()
            optimizer = torch.optim.AdamW(projector.parameters(), lr=1e-3)
            text = batch["text_embedding"].to(DEVICE)
            target = batch["volume"].to(DEVICE)
            with torch.no_grad():
                target_latent = autoencoder.encoder(target)
            optimizer.zero_grad(set_to_none=True)
            scaler = torch.amp.GradScaler(
                "cuda", enabled=DEVICE.type == "cuda" and dtype == torch.float16
            )
            with torch.autocast(
                device_type=DEVICE.type, dtype=dtype,
                enabled=DEVICE.type == "cuda" and dtype != torch.float32,
            ):
                result = compute_loss(
                    projector, autoencoder, text, target, target_latent,
                    {name: 1.0 for name in COMPONENT_NAMES},
                )
                selected = result.components[component]
            scaler.scale(selected).backward()
            scaler.unscale_(optimizer)
            gradients = [
                parameter.grad.detach().float()
                for parameter in projector.parameters() if parameter.grad is not None
            ]
            finite = bool(torch.isfinite(selected.detach().float())) and all(
                bool(torch.isfinite(gradient).all()) for gradient in gradients
            )
            rows.append({
                "dtype": str(dtype), "component": component,
                "loss": float(selected.detach()), "finite": finite,
                "gradient_norm": float(torch.sqrt(sum(g.square().sum() for g in gradients))),
                "batch_source": "train",
            })
    return rows

def synthetic_loss_unit_tests():
    generator = torch.Generator().manual_seed(123)
    prediction = torch.randn(4, 1, 5, 6, 7, generator=generator, requires_grad=True)
    target = torch.zeros_like(prediction)
    target[0, 0, 1, 2, 3] = 1.0
    target[1, 0, 2:4, 1, 1] = torch.tensor([0.2, 0.9])
    target[2, 0, :2, :2, :2] = 0.5
    # Example 3 intentionally remains all-background.
    components = spatial_loss_components(prediction, target)
    assert set(components) == set(SPATIAL_COMPONENTS)
    rows = []
    for name, value in components.items():
        gradient = torch.autograd.grad(value, prediction, retain_graph=True)[0]
        finite = bool(torch.isfinite(value) and torch.isfinite(gradient).all())
        assert finite, f"{name} produced non-finite synthetic gradients"
        assert float(gradient.abs().sum()) > 0, f"{name} produced a zero synthetic gradient"
        rows.append({"test": "sparse_synthetic", "component": name, "finite": finite})
    negative_prediction = torch.full_like(target, -0.5, requires_grad=True)
    dense = spatial_loss_components(negative_prediction, target)["dense"]
    dense.backward()
    assert negative_prediction.grad is not None and torch.isfinite(negative_prediction.grad).all()
    rows.append({"test": "raw_negative_dense_mse", "component": "dense", "finite": True})
    return rows

SYNTHETIC_TEST_ROWS = synthetic_loss_unit_tests() if RUN_UNIT_TESTS else []
print("Synthetic differentiable-loss tests:", len(SYNTHETIC_TEST_ROWS), "passed")


## 7. Evaluation, semantic retrieval, checkpoints, and training

Spatial overlap metrics follow the repository convention: non-finite predictions are made finite and clamped to `[0,1]` only inside metrics. Training losses never clamp. Generated-map semantic normalized recall AUC uses the released branch-matched Stage 3 contrastive encoder; generated maps are finite/clamped for that evaluator, and semantic positives are the exact source plus its nearest cached-SPECTER2 neighbors. A deterministic bounded prefix prevents quadratic retrieval memory growth, and the evaluated count is recorded.


In [ ]:
@dataclass
class EvaluationResult:
    summary: dict
    examples: list
    prediction_voxel_sample: Tensor
    target_voxel_sample: Tensor
    n: int

def scalar_latent_metrics(target, prediction):
    target = target.float()
    prediction = prediction.float()
    residual = target - prediction
    target_variance = target.var(dim=0, unbiased=False).sum().clamp_min(EPSILON)
    prediction_variance = prediction.var(dim=0, unbiased=False).sum()
    target_norm = target.norm(dim=1).mean().clamp_min(EPSILON)
    return {
        "raw_latent_mse": float(F.mse_loss(prediction, target)),
        "latent_variance_ratio": float(prediction_variance / target_variance),
        "latent_norm_ratio": float(prediction.norm(dim=1).mean() / target_norm),
        "explained_variance": float(1 - residual.var(dim=0, unbiased=False).sum() / target_variance),
    }

def extra_spatial_metrics(prediction, target):
    raw_pred = _flat_fp32(prediction)
    raw_target = _flat_fp32(target)
    pred = raw_pred.clamp(0, 1)
    truth = raw_target.clamp(0, 1)
    squared = (pred - truth).square()
    foreground = truth > 0
    background = ~foreground
    dense = squared.mean(dim=1)
    fg = _masked_per_example_mean(squared, foreground, dense)
    bg = _masked_per_example_mean(squared, background, dense)
    recall = _masked_per_example_mean((raw_pred > 0).float(), foreground, torch.zeros_like(dense))
    return {
        "foreground_mse": float(fg.mean()),
        "background_mse": float(bg.mean()),
        "positive_voxel_recall": float(recall.mean()),
        "target_voxel_mean": float(raw_target.mean()),
        "prediction_voxel_mean": float(raw_pred.mean()),
        "prediction_voxel_abs_mean": float(raw_pred.abs().mean()),
        "prediction_voxel_min": float(raw_pred.min()),
        "prediction_voxel_max": float(raw_pred.max()),
        "negative_fraction_before_clamping": float((raw_pred < 0).float().mean()),
        "target_positive_fraction": float(foreground.float().mean()),
    }

def semantic_metrics(semantic_model, brain_embeddings, text_embeddings, raw_text_embeddings, ids):
    n_examples = sum(len(value) for value in brain_embeddings)
    if n_examples < 2:
        return {"semantic_normalized_auc": float("nan"), "semantic_n": n_examples}
    # Avoid the degenerate case where every candidate is declared a
    # semantic positive in small FAST_ABLATION cohorts.
    effective_neighbors = max(0, min(SEMANTIC_NEIGHBORS, n_examples - 2))
    metrics, _ = evaluate_semantic_neighbor_retrieval(
        torch.cat(brain_embeddings),
        torch.cat(text_embeddings),
        ids,
        neighbor_text_embeddings=torch.cat(raw_text_embeddings),
        n_neighbors=effective_neighbors,
    )
    return {
        "semantic_normalized_auc": float(metrics["semantic_normalized_k_recall_curve_auc"]),
        "semantic_n": int(metrics["n_queries"]),
    }

def evaluate(
    projector, autoencoder, semantic_model, dataset, lookup, weights,
    *, split, gradient_batch=None, max_batches=None,
):
    projector.eval()
    autoencoder.eval()
    semantic_model.eval()
    totals = {}
    n = 0
    target_latents, prediction_latents = [], []
    examples = []
    prediction_samples, target_samples = [], []
    semantic_brain, semantic_text, semantic_raw_text, semantic_ids = [], [], [], []
    loader = make_loader(dataset, lookup, batch_size=EVAL_BATCH_SIZE, shuffle=False, seed=SEED)
    with torch.no_grad():
        for batch_index, batch in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            target = batch["volume"].to(DEVICE, non_blocking=True)
            text = batch["text_embedding"].to(DEVICE, non_blocking=True)
            target_latent = autoencoder.encoder(target)
            prediction_latent = projector(text)
            prediction = autoencoder.decoder(prediction_latent)
            components = {
                "latent": F.mse_loss(prediction_latent.float(), target_latent.float()),
                **spatial_loss_components(prediction, target),
            }
            weighted = {name: components[name] * weights[name] for name in COMPONENT_NAMES}
            batch_metrics = {
                **{f"raw_{name}_loss": float(value) for name, value in components.items()},
                **{f"weighted_{name}_contribution": float(weighted[name]) for name in COMPONENT_NAMES},
                "loss": float(sum(weighted.values())),
                **reconstruction_metrics(prediction, target),
                **extra_spatial_metrics(prediction, target),
            }
            batch_n = len(target)
            for name, value in batch_metrics.items():
                totals[name] = totals.get(name, 0.0) + float(value) * batch_n
            target_latents.append(target_latent.float().cpu())
            prediction_latents.append(prediction_latent.float().cpu())
            n += batch_n

            support = target.flatten(1).gt(0).float().mean(dim=1)
            batch_examples = []
            for index in range(batch_n):
                batch_examples.append({
                    "map_id": str(batch["map_id"][index]),
                    "text_id": str(batch["text_id"][index]),
                    "source": str(batch["source"][index]),
                    "target_positive_fraction": float(support[index]),
                    "prediction": prediction[index].float().cpu(),
                    "target": target[index].float().cpu(),
                })
            # Retain bounded sparse/dense extremes instead of every
            # split volume, which can exhaust host memory.
            examples.extend(batch_examples)
            examples = sorted(
                examples, key=lambda row: row["target_positive_fraction"]
            )
            keep = max(1, EXAMPLES_PER_RUN // 2)
            examples = examples[:keep] + examples[-keep:]
            remaining = DISTRIBUTION_SAMPLE_VOXELS - sum(x.numel() for x in prediction_samples)
            if remaining > 0:
                flat_p = prediction.detach().float().cpu().flatten()
                flat_t = target.detach().float().cpu().flatten()
                take = min(remaining, flat_p.numel())
                step = max(1, flat_p.numel() // take)
                prediction_samples.append(flat_p[::step][:take])
                target_samples.append(flat_t[::step][:take])

            semantic_count = sum(len(value) for value in semantic_brain)
            remaining_semantic = max(0, SEMANTIC_MAX_EXAMPLES - semantic_count)
            if remaining_semantic:
                take = min(batch_n, remaining_semantic)
                semantic_prediction = torch.nan_to_num(
                    prediction[:take].float(), nan=0.0, posinf=1.0, neginf=0.0
                ).clamp(0, 1)
                semantic_brain.append(semantic_model.encode_brain(semantic_prediction).float().cpu())
                semantic_text.append(semantic_model.encode_text(text[:take]).float().cpu())
                semantic_raw_text.append(text[:take].float().cpu())
                semantic_ids.extend(str(value) for value in batch["map_id"][:take])
    if not n:
        raise RuntimeError(f"{split} evaluation produced no examples")
    summary = {name: value / n for name, value in totals.items()}
    summary.update(scalar_latent_metrics(torch.cat(target_latents), torch.cat(prediction_latents)))
    summary.update(semantic_metrics(
        semantic_model, semantic_brain, semantic_text, semantic_raw_text, semantic_ids
    ))
    if gradient_batch is not None:
        gradients = sampled_component_gradients(
            projector, autoencoder, gradient_batch, weights
        )
        summary.update({f"sample_gradient_norm_{name}": value for name, value in gradients.items()})
    examples = sorted(examples, key=lambda row: row["target_positive_fraction"])
    selected = examples[: EXAMPLES_PER_RUN // 2] + examples[-(EXAMPLES_PER_RUN - EXAMPLES_PER_RUN // 2):]
    return EvaluationResult(
        summary=summary,
        examples=selected,
        prediction_voxel_sample=torch.cat(prediction_samples),
        target_voxel_sample=torch.cat(target_samples),
        n=n,
    )

def sampled_component_gradients(projector, autoencoder, batch, weights):
    projector.train()
    text = batch["text_embedding"].to(DEVICE)
    target = batch["volume"].to(DEVICE)
    with torch.no_grad():
        target_latent = autoencoder.encoder(target)
    with torch.autocast(
        device_type=DEVICE.type, dtype=MIXED_PRECISION_DTYPE,
        enabled=AMP and DEVICE.type == "cuda" and MIXED_PRECISION_DTYPE != torch.float32,
    ):
        result = compute_loss(projector, autoencoder, text, target, target_latent, weights)
    parameters = tuple(projector.parameters())
    values = {}
    losses = [result.components[name] for name in COMPONENT_NAMES] + [result.total]
    names = list(COMPONENT_NAMES) + ["total"]
    for index, (name, loss) in enumerate(zip(names, losses, strict=True)):
        values[name] = parameter_gradient_norm(
            loss, parameters, retain_graph=index < len(losses) - 1
        )
    projector.eval()
    return values

class SpatialCheckpointManager(AblationCheckpointManager):
    def update_foreground_mse(
        self, value, projector, optimizer, *, epoch, metrics, early_stopping
    ):
        value = float(value)
        current = (
            self._manifest().get("checkpoints", {}).get("foreground_mse") or {}
        ).get("value")
        if not math.isfinite(value) or (current is not None and value >= float(current)):
            return None
        return self._save(
            "best_validation_foreground_mse.pt", "foreground_mse",
            projector, optimizer, epoch=epoch, metrics=metrics,
            metric="val_foreground_mse", value=value, scheduler=None,
            early_stopping=early_stopping,
        )

def read_csv_rows(path):
    if not Path(path).exists():
        return []
    with Path(path).open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))

def train_epoch(projector, autoencoder, dataset, lookup, train_latents, optimizer, weights, epoch):
    projector.train()
    autoencoder.eval()
    parameters = tuple(projector.parameters())
    autocast_enabled = AMP and DEVICE.type == "cuda" and MIXED_PRECISION_DTYPE != torch.float32
    scaler = torch.amp.GradScaler(
        "cuda", enabled=autocast_enabled and MIXED_PRECISION_DTYPE == torch.float16
    )
    totals, n = {}, 0
    loader = make_loader(dataset, lookup, batch_size=BATCH_SIZE, shuffle=True, seed=SEED + epoch)
    for batch_index, batch in enumerate(loader):
        if MAX_TRAIN_BATCHES is not None and batch_index >= MAX_TRAIN_BATCHES:
            break
        text = batch["text_embedding"].to(DEVICE, non_blocking=True)
        target = batch["volume"].to(DEVICE, non_blocking=True)
        indices = torch.as_tensor(batch["dataset_index"], dtype=torch.long)
        target_latent = train_latents.index_select(0, indices).to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(
            device_type=DEVICE.type, dtype=MIXED_PRECISION_DTYPE, enabled=autocast_enabled
        ):
            result = compute_loss(projector, autoencoder, text, target, target_latent, weights)
        if scaler.is_enabled():
            scaler.scale(result.total).backward()
            scaler.unscale_(optimizer)
        else:
            result.total.backward()
        gradient_norm = float(torch.sqrt(sum(
            parameter.grad.detach().float().square().sum()
            for parameter in parameters if parameter.grad is not None
        )))
        before = [parameter.detach().clone() for parameter in parameters]
        if GRADIENT_CLIP is not None:
            torch.nn.utils.clip_grad_norm_(parameters, GRADIENT_CLIP)
        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        update_norm = float(torch.sqrt(sum(
            (parameter.detach() - previous).float().square().sum()
            for parameter, previous in zip(parameters, before, strict=True)
        )))
        batch_metrics = {
            **{f"raw_{name}_loss": float(value.detach()) for name, value in result.components.items()},
            **{f"weighted_{name}_contribution": float(result.weighted[name].detach()) for name in COMPONENT_NAMES},
            "loss": float(result.total.detach()),
            "total_gradient_norm": gradient_norm,
            "parameter_update_norm": update_norm,
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
        }
        batch_n = len(target)
        for name, value in batch_metrics.items():
            totals[name] = totals.get(name, 0.0) + value * batch_n
        n += batch_n
    if not n:
        raise RuntimeError("Training produced no batches")
    return {name: value / n for name, value in totals.items()}, n

def train_run(
    run_dir, spec, autoencoder, semantic_model, provider, lookup,
    train_latents, provenance, loss_spec, calibration, val_gradient_batch,
):
    run_dir.mkdir(parents=True, exist_ok=True)
    weights = effective_weights(loss_spec, calibration)
    effective = {
        "branch": spec, "loss": loss_spec, "effective_weights": weights,
        "auto_gradient_balance": AUTO_GRADIENT_BALANCE,
        "gradient_balance_factor": GRADIENT_BALANCE_FACTOR,
        "architecture": {"name": "GenerativeTextToAELatent", "layers": [768, 512, "ReLU", 384]},
        "raw_latent_mse_weight": 1.0, "raw_decoder_training": True,
        "optimizer": "AdamW", "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY, "scheduler": SCHEDULER,
        "batch_size": BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
        "epochs": EPOCHS, "validation_interval_epochs": VALIDATE_EVERY_EPOCHS,
        "gradient_clip": GRADIENT_CLIP,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "amp": AMP, "amp_dtype": str(MIXED_PRECISION_DTYPE),
        "seed": SEED, "projector_seed": PROJECTOR_SEED,
        "test_used_for_selection": False,
        "fast_ablation": FAST_ABLATION,
    }
    binding = {
        "autoencoder": provenance["autoencoder"],
        "text_cache": provenance["text_cache"],
        "splits": provenance["splits"],
        "branch": provenance["branch"],
        "projector": provenance["projector"],
        "loss_sha256": sha256_value(loss_spec),
    }
    atomic_write_json(run_dir / "effective_config.json", effective)
    atomic_write_json(run_dir / "provenance.json", provenance)
    atomic_write_json(run_dir / "gradient_calibration.json", {
        **calibration, "effective_weights": weights, "loss_spec": loss_spec
    })
    seed_everything(PROJECTOR_SEED)
    projector = GenerativeTextToAELatent(768, 512, 384).to(DEVICE)
    optimizer = torch.optim.AdamW(
        projector.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    manager = SpatialCheckpointManager(
        run_dir, binding=binding, architecture=effective["architecture"], config=effective
    )
    resumed = manager.resume(projector, optimizer, map_location=DEVICE)
    start_epoch = 1 if resumed is None else int(resumed["epoch"]) + 1
    early = dict((resumed or {}).get("early_stopping") or {})
    early_best = float(early.get("best", -float("inf")))
    stale_epochs = int(early.get("stale_epochs", 0))
    history = [
        row for row in read_csv_rows(run_dir / "training_history.csv")
        if int(row["epoch"]) < start_epoch
    ]
    validation_rows = [
        row for row in read_csv_rows(run_dir / "validation_metrics.csv")
        if int(row["epoch"]) < start_epoch
    ]
    for epoch in range(start_epoch, EPOCHS + 1):
        train_metrics, train_n = train_epoch(
            projector, autoencoder, provider.train, lookup,
            train_latents, optimizer, weights, epoch,
        )
        validation = evaluate(
            projector, autoencoder, semantic_model, provider.val, lookup, weights,
            split="val", gradient_batch=val_gradient_batch, max_batches=MAX_EVAL_BATCHES,
        )
        validation.summary.update({
            "total_gradient_norm": train_metrics["total_gradient_norm"],
            "parameter_update_norm": train_metrics["parameter_update_norm"],
            "learning_rate": train_metrics["learning_rate"],
        })
        history.append({"epoch": epoch, "n": train_n, **{f"train_{k}": v for k, v in train_metrics.items()}})
        validation_row = {"epoch": epoch, "n": validation.n, **{f"val_{k}": v for k, v in validation.summary.items()}}
        validation_rows.append(validation_row)
        atomic_write_csv(run_dir / "training_history.csv", history)
        atomic_write_csv(run_dir / "validation_metrics.csv", validation_rows)
        top5 = float(validation.summary["top5_dice"])
        if top5 > early_best + EARLY_STOPPING_MIN_DELTA:
            early_best, stale_epochs = top5, 0
        else:
            stale_epochs += 1
        early_state = {"best": early_best, "stale_epochs": stale_epochs}
        checkpoint_metrics = {**history[-1], **validation_row}
        manager.update_best(
            "val_top5_dice", top5, projector, optimizer,
            epoch=epoch, metrics=checkpoint_metrics, early_stopping=early_state,
        )
        manager.update_best(
            "val_spatial_corr", float(validation.summary["spatial_corr"]),
            projector, optimizer, epoch=epoch, metrics=checkpoint_metrics,
            early_stopping=early_state,
        )
        manager.update_foreground_mse(
            float(validation.summary["foreground_mse"]), projector, optimizer,
            epoch=epoch, metrics=checkpoint_metrics, early_stopping=early_state,
        )
        manager.update_best(
            "val_semantic_normalized_auc",
            float(validation.summary["semantic_normalized_auc"]),
            projector, optimizer, epoch=epoch, metrics=checkpoint_metrics,
            early_stopping=early_state,
        )
        if not math.isfinite(float(validation.summary["semantic_normalized_auc"])):
            raise FloatingPointError(
                "Validation semantic normalized AUC is non-finite; "
                "refusing to omit the required semantic checkpoint"
            )
        manager.save_last(
            projector, optimizer, epoch=epoch,
            metrics=checkpoint_metrics, early_stopping=early_state,
        )
        if EARLY_STOPPING_PATIENCE is not None and stale_epochs >= EARLY_STOPPING_PATIENCE:
            break
    return {
        "projector": projector, "optimizer": optimizer, "manager": manager,
        "weights": weights, "epochs_completed": int(history[-1]["epoch"]),
    }


## 8. Tiny 32-example controls and plots

The correctness-audit utility is run unchanged for reconstruction-only, current combined, and foreground-combined paired controls, plus a shuffled-pair combined negative control. AE-ceiling gaps and independent BF16/FP16 finite-gradient tests are recorded.


In [ ]:
def run_tiny_controls(branch_dir, autoencoder, train_batch, amp_rows):
    controls_dir = branch_dir / "tiny_controls"
    controls_dir.mkdir(parents=True, exist_ok=True)
    target = train_batch["volume"][:TINY_N].to(DEVICE)
    text = train_batch["text_embedding"][:TINY_N].to(DEVICE)
    results = []
    for loss_mode, shuffled in [
        ("reconstruction_only", False),
        ("combined", False),
        ("foreground_combined", False),
        ("combined", True),
    ]:
        seed_everything(PROJECTOR_SEED)
        model = CNNTextToBrainModel(
            GenerativeTextToAELatent().to(DEVICE), autoencoder.to(DEVICE)
        )
        result = tiny_overfit_projector(
            model, text, target, steps=TINY_STEPS, learning_rate=TINY_LR,
            loss_mode=loss_mode, shuffled_pairing=shuffled,
            report_every=max(1, TINY_STEPS // 20), foreground_weight=4.0,
        )
        final = result["final"]
        final["top5_dice_ceiling_gap"] = final["ceiling_top5_dice"] - final["top5_dice"]
        final["spatial_corr_ceiling_gap"] = final["ceiling_spatial_corr"] - final["spatial_corr"]
        result["approaches_ae_ceiling"] = (
            final["top5_dice_ceiling_gap"] <= 0.05
            and final["spatial_corr_ceiling_gap"] <= 0.05
        )
        results.append(result)
    report = {
        "n": len(target), "results": results,
        "amp_gradients_finite": all(bool(row["finite"]) for row in amp_rows),
        "amp_dtypes_tested": sorted({row["dtype"] for row in amp_rows}),
    }
    atomic_write_json(controls_dir / "tiny_control_report.json", report)
    atomic_write_csv(
        controls_dir / "tiny_control_history.csv",
        [
            {
                "loss_mode": result["loss_mode"],
                "shuffled_pairing": result["shuffled_pairing"],
                **row,
            }
            for result in results for row in result["history"]
        ],
    )
    return report

def save_run_plots(run_dir, evaluation):
    plots = run_dir / "plots"
    examples_dir = run_dir / "examples"
    plots.mkdir(exist_ok=True)
    examples_dir.mkdir(exist_ok=True)
    train = pd.read_csv(run_dir / "training_history.csv")
    val = pd.read_csv(run_dir / "validation_metrics.csv")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    raw_cols = [f"train_raw_{name}_loss" for name in COMPONENT_NAMES]
    for column in raw_cols:
        if column in train:
            axes[0].plot(train["epoch"], train[column], label=column.removeprefix("train_raw_").removesuffix("_loss"))
    axes[0].set_title("Training raw loss components")
    axes[0].set_yscale("log")
    axes[0].legend(fontsize=7, ncol=2)
    weighted_cols = [f"val_weighted_{name}_contribution" for name in COMPONENT_NAMES]
    for column in weighted_cols:
        if column in val and np.any(np.asarray(val[column], dtype=float) != 0):
            axes[1].plot(val["epoch"], val[column], label=column.removeprefix("val_weighted_").removesuffix("_contribution"))
    axes[1].set_title("Validation weighted contributions")
    axes[1].set_yscale("symlog", linthresh=1e-8)
    axes[1].legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(plots / "loss_component_curves.png", dpi=160)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(9, 4))
    for name in [*COMPONENT_NAMES, "total"]:
        column = f"val_sample_gradient_norm_{name}"
        if column in val:
            ax.plot(val["epoch"], val[column], label=name)
    ax.set_yscale("log")
    ax.set_title("Sampled validation component-gradient norms")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(plots / "component_gradient_curves.png", dpi=160)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(val["epoch"], val["val_foreground_mse"], label="foreground")
    axes[0].plot(val["epoch"], val["val_background_mse"], label="background")
    axes[0].set_title("Foreground versus background error")
    axes[0].legend()
    axes[1].plot(val["epoch"], val["val_latent_variance_ratio"])
    axes[1].axhline(1, color="black", ls="--")
    axes[1].set_title("Raw latent variance ratio")
    fig.tight_layout()
    fig.savefig(plots / "foreground_background_and_latent_variance.png", dpi=160)
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(evaluation.target_voxel_sample.numpy(), bins=100, alpha=0.6, density=True, label="target")
    axes[0].hist(evaluation.prediction_voxel_sample.numpy(), bins=100, alpha=0.6, density=True, label="prediction raw")
    axes[0].set_yscale("log")
    axes[0].set_title("Target versus generated voxels")
    axes[0].legend()
    pred = evaluation.prediction_voxel_sample.numpy()
    axes[1].hist(pred, bins=100, density=True)
    axes[1].axvline(0, color="black", ls="--")
    axes[1].set_title("Prediction amplitude before clamping")
    fig.tight_layout()
    fig.savefig(plots / "voxel_distributions_and_raw_amplitude.png", dpi=160)
    plt.close(fig)

    torch.save({"items": evaluation.examples}, examples_dir / "selected_examples.pt")
    if evaluation.examples:
        fig, axes = plt.subplots(len(evaluation.examples), 2, figsize=(8, 3 * len(evaluation.examples)), squeeze=False)
        for row, example in enumerate(evaluation.examples):
            target = example["target"].squeeze()
            prediction = example["prediction"].squeeze()
            energy = target.abs().sum(dim=(0, 1))
            z = int(energy.argmax())
            axes[row, 0].imshow(target[:, :, z], cmap="hot")
            axes[row, 0].set_title(f"target {example['map_id']} | support={example['target_positive_fraction']:.3f}")
            axes[row, 1].imshow(prediction[:, :, z], cmap="coolwarm")
            axes[row, 1].set_title("raw prediction")
            axes[row, 0].axis("off")
            axes[row, 1].axis("off")
        fig.tight_layout()
        fig.savefig(examples_dir / "sparse_and_dense_examples.png", dpi=160)
        plt.close(fig)

def evaluate_all_checkpoints(run_dir, result, autoencoder, semantic_model, provider, lookup):
    manifest = json.loads((run_dir / "checkpoint_manifest.json").read_text())
    rows = []
    top5_evaluation = None
    for role, record in manifest["checkpoints"].items():
        result["manager"].resume(
            result["projector"], path=record["path"], map_location=DEVICE
        )
        evaluation = evaluate(
            result["projector"], autoencoder, semantic_model, provider.test,
            lookup, result["weights"], split="test", max_batches=MAX_EVAL_BATCHES,
        )
        rows.append({
            "checkpoint_role": role, "checkpoint_epoch": record["epoch"],
            "n": evaluation.n, **evaluation.summary,
        })
        if role == "top5_dice":
            top5_evaluation = evaluation
    atomic_write_csv(run_dir / "test_metrics.csv", rows)
    if top5_evaluation is None:
        raise RuntimeError("No validation-selected top-5 Dice checkpoint exists")
    top5_record = manifest["checkpoints"]["top5_dice"]
    result["manager"].resume(
        result["projector"], path=top5_record["path"], map_location=DEVICE
    )
    selected_val = evaluate(
        result["projector"], autoencoder, semantic_model, provider.val,
        lookup, result["weights"], split="val", max_batches=MAX_EVAL_BATCHES,
    )
    save_run_plots(run_dir, selected_val)
    return rows, selected_val


## 9. Run selected branches

This cell is resume-safe at the experiment and individual-run levels. It writes requested artifacts incrementally to Drive. The ordered train latent cache is accepted only when both the split fingerprint and frozen encoder checksum match.


In [ ]:
ROOT_CONFIG = {
    "branches_to_run": BRANCHES_TO_RUN,
    "fast_ablation": FAST_ABLATION,
    "loss_specs": LOSS_SPECS,
    "seed": SEED, "projector_seed": PROJECTOR_SEED,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "gradient_clip": GRADIENT_CLIP,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "scheduler": SCHEDULER, "amp": AMP,
    "amp_dtype": str(MIXED_PRECISION_DTYPE),
    "auto_gradient_balance": AUTO_GRADIENT_BALANCE,
    "gradient_balance_factor": GRADIENT_BALANCE_FACTOR,
    "foreground_lambdas": LAMBDA_FG_VALUES,
    "positive_lambda": LAMBDA_POSITIVE,
    "dice_lambda": LAMBDA_DICE,
    "correlation_lambda": LAMBDA_CORR,
    "hybrid_weights": HYBRID_WEIGHTS,
    "softplus_beta": SOFTPLUS_BETA,
    "soft_top5_temperature": SOFT_TOP5_TEMPERATURE,
    "epsilon": EPSILON,
    "semantic_max_examples": SEMANTIC_MAX_EXAMPLES,
    "data_limit": FULL_DATA_LIMIT,
    "test_used_for_selection": False,
}
atomic_write_json(EXPERIMENT_ROOT / "effective_config.json", ROOT_CONFIG)
atomic_write_json(EXPERIMENT_ROOT / "provenance.json", {
    "environment": ENVIRONMENT, "branches": {}, "resolved_commit": RESOLVED_COMMIT
})
atomic_write_csv(EXPERIMENT_ROOT / "synthetic_loss_tests.csv", SYNTHETIC_TEST_ROWS)

lookup = AtlasFreeTextEmbeddingLookup.published()
summary_rows = read_csv_rows(EXPERIMENT_ROOT / "per_run_summary.csv")
all_calibrations = {}
all_provenance = {}
domain_fixed_batch_ids = {}

for branch_name in BRANCHES_TO_RUN:
    print(f"\n===== {branch_name} =====")
    branch_dir = EXPERIMENT_ROOT / branch_name
    branch_dir.mkdir(parents=True, exist_ok=True)
    audit_dir = branch_dir / "provenance_audits"
    audit_dir.mkdir(exist_ok=True)
    spec, ae_path, autoencoder, provider, semantic_model = load_branch_resources(branch_name)
    provenance = build_provenance(
        spec, ae_path, autoencoder, provider, lookup, audit_dir
    )
    all_provenance[branch_name] = provenance
    atomic_write_json(branch_dir / "provenance.json", provenance)

    train_latents_path = branch_dir / "training_target_latents.pt"
    train_fingerprint = provenance["splits"]["train"]["ordered_rows_sha256"]
    encoder_hash = provenance["autoencoder"]["encoder_state_sha256"]
    if train_latents_path.exists():
        cached = torch.load(train_latents_path, map_location="cpu", weights_only=True)
        if cached["split_sha256"] != train_fingerprint:
            raise ValueError("Training latent split fingerprint mismatch")
        if cached["encoder_state_sha256"] != encoder_hash:
            raise ValueError("Training latent encoder checksum mismatch")
        train_latents = cached["latents"]
    else:
        train_latents = encode_stage1_latents(
            autoencoder, provider.train, lookup, device=DEVICE,
            batch_size=EVAL_BATCH_SIZE, num_workers=NUM_WORKERS,
        )
        torch.save(
            {
                "latents": train_latents, "split_sha256": train_fingerprint,
                "encoder_state_sha256": encoder_hash,
            },
            train_latents_path,
        )
    if len(train_latents) != len(provider.train):
        raise RuntimeError("Ordered train latent count mismatch")

    train_gradient_batch = fixed_batch(provider.train, lookup)
    val_gradient_batch = fixed_batch(provider.val, lookup)
    fixed_ids = [str(value) for value in train_gradient_batch["map_id"]]
    previous_ids = domain_fixed_batch_ids.setdefault(spec["domain"], fixed_ids)
    if previous_ids != fixed_ids:
        raise RuntimeError(
            f"Fixed calibration batch changed within domain {spec['domain']}"
        )
    calibration = gradient_calibration(autoencoder, train_gradient_batch)
    all_calibrations[branch_name] = calibration
    atomic_write_json(branch_dir / "gradient_calibration.json", calibration)

    dtype_tests = [torch.float32]
    if DEVICE.type == "cuda":
        dtype_tests.append(torch.float16)
        if torch.cuda.is_bf16_supported():
            dtype_tests.append(torch.bfloat16)
    amp_rows = amp_component_tests(autoencoder, train_gradient_batch, dtype_tests)
    if not all(row["finite"] for row in amp_rows):
        failures = [row for row in amp_rows if not row["finite"]]
        raise FloatingPointError(f"Non-finite real-batch AMP gradients: {failures}")
    atomic_write_csv(branch_dir / "amp_loss_tests.csv", amp_rows)

    if RUN_UNIT_TESTS:
        real_rows = [
            {"test": "real_batch_amp", **row} for row in amp_rows
        ]
        atomic_write_csv(branch_dir / "differentiable_loss_tests.csv", [
            *SYNTHETIC_TEST_ROWS, *real_rows
        ])

    if RUN_TINY_CONTROLS:
        tiny = run_tiny_controls(
            branch_dir, autoencoder, train_gradient_batch, amp_rows
        )
        print("Tiny controls complete; AMP finite:", tiny["amp_gradients_finite"])

    if RUN_FULL_TRAINING:
        for loss_spec in LOSS_SPECS:
            run_name = loss_spec["run_name"]
            print(f"--- {branch_name} / {run_name} ---")
            run_dir = branch_dir / run_name
            result = train_run(
                run_dir, spec, autoencoder, semantic_model, provider, lookup,
                train_latents, provenance, loss_spec, calibration, val_gradient_batch,
            )
            test_rows, selected_val = evaluate_all_checkpoints(
                run_dir, result, autoencoder, semantic_model, provider, lookup
            )
            selected_test = next(
                row for row in test_rows if row["checkpoint_role"] == "top5_dice"
            )
            row = {
                "branch": branch_name, "domain": spec["domain"],
                "stage1": spec["stage1"], "objective": loss_spec["objective"],
                "run_name": run_name, "epochs_completed": result["epochs_completed"],
                **{f"weight_{key}": value for key, value in result["weights"].items()},
                **{f"val_{key}": value for key, value in selected_val.summary.items()},
                **{f"test_{key}": value for key, value in selected_test.items()
                   if key not in {"checkpoint_role"}},
                "run_dir": str(run_dir),
            }
            atomic_write_csv(run_dir / "per_run_summary.csv", [row])
            summary_rows = [
                existing for existing in summary_rows
                if not (
                    str(existing["branch"]) == branch_name
                    and str(existing["run_name"]) == run_name
                )
            ]
            summary_rows.append(row)
            atomic_write_csv(EXPERIMENT_ROOT / "per_run_summary.csv", summary_rows)
            atomic_write_csv(EXPERIMENT_ROOT / "all_runs_comparison.csv", summary_rows)

    del autoencoder, semantic_model, provider, train_latents
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

atomic_write_json(EXPERIMENT_ROOT / "gradient_calibration.json", all_calibrations)
atomic_write_json(EXPERIMENT_ROOT / "provenance.json", {
    "environment": ENVIRONMENT, "branches": all_provenance,
    "resolved_commit": RESOLVED_COMMIT,
})
print("Experiment outputs:", EXPERIMENT_ROOT)


## 10. Cross-run plots and final causal report

Conclusions are based on validation-selected top-5 checkpoints and validation deltas. Test metrics are reported but never used to choose a loss or checkpoint. “Causal contribution” here means an intervention on spatial supervision, with architecture/data/latent resources fixed, systematically improves sparse-map collapse indicators over the dense-MSE control.


In [ ]:
def build_final_outputs(root):
    comparison_path = root / "all_runs_comparison.csv"
    if not comparison_path.exists():
        report = "# Stage 4 sparse spatial-loss ablation report\n\nNo completed full-training runs are available."
        (root / "final_report.md").write_text(report)
        return
    frame = pd.read_csv(comparison_path)
    baselines = frame[frame["objective"] == "baseline_dense_mse"].set_index("branch")
    metrics = [
        "val_top5_dice", "val_spatial_corr", "val_foreground_mse",
        "val_semantic_normalized_auc", "val_latent_variance_ratio",
        "val_latent_norm_ratio", "val_prediction_voxel_abs_mean",
    ]
    for metric in metrics:
        frame[f"delta_{metric}_vs_dense"] = frame.apply(
            lambda row: row[metric] - baselines.loc[row["branch"], metric],
            axis=1,
        )
    atomic_write_csv(root / "all_runs_comparison.csv", frame.to_dict("records"))

    plots = root / "plots"
    plots.mkdir(exist_ok=True)
    examples_root = root / "examples"
    examples_root.mkdir(exist_ok=True)
    atomic_write_json(
        examples_root / "example_manifest.json",
        {
            "runs": [
                {
                    "branch": str(row.branch),
                    "run_name": str(row.run_name),
                    "examples": str(Path(row.run_dir) / "examples" / "selected_examples.pt"),
                    "plot": str(Path(row.run_dir) / "examples" / "sparse_and_dense_examples.png"),
                }
                for row in frame.itertuples()
            ]
        },
    )
    alternatives = frame[frame.objective != "baseline_dense_mse"]
    for metric, filename, title in [
        ("val_top5_dice", "top5_dice_by_domain.png", "Top-5 Dice by domain"),
        ("val_spatial_corr", "spatial_correlation_by_domain.png", "Spatial correlation by domain"),
    ]:
        fig, ax = plt.subplots(figsize=(14, 5))
        pivot = frame.pivot_table(index="run_name", columns="domain", values=metric, aggfunc="mean")
        pivot.plot(kind="bar", ax=ax)
        ax.set_title(title)
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", labelrotation=60)
        fig.tight_layout()
        fig.savefig(plots / filename, dpi=160)
        plt.close(fig)

    sparse = alternatives[alternatives.domain.isin(["pubmed", "nilearn"])]
    analysis_frame = sparse if len(sparse) else alternatives
    analysis_scope = "PubMed and Nilearn" if len(sparse) else "selected non-sparse branches (no PubMed/Nilearn branch was run)"
    by_objective = analysis_frame.groupby(["objective", "run_name"], as_index=False).agg(
        mean_top5_delta=("delta_val_top5_dice_vs_dense", "mean"),
        mean_corr_delta=("delta_val_spatial_corr_vs_dense", "mean"),
        mean_foreground_mse_delta=("delta_val_foreground_mse_vs_dense", "mean"),
        mean_semantic_delta=("delta_val_semantic_normalized_auc_vs_dense", "mean"),
        mean_amplitude_delta=("delta_val_prediction_voxel_abs_mean_vs_dense", "mean"),
        mean_latent_variance_delta=("delta_val_latent_variance_ratio_vs_dense", "mean"),
    )
    best = by_objective.sort_values("mean_top5_delta", ascending=False).iloc[0]
    neurovault = alternatives[alternatives.domain == "neurovault"]
    sparse_best_gain = float(best["mean_top5_delta"])
    nv_best_gain = (
        float(neurovault.groupby("run_name")["delta_val_top5_dice_vs_dense"].mean().max())
        if len(neurovault) else float("nan")
    )
    causal = (
        "not evaluated for sparse maps"
        if not len(sparse)
        else "supported"
        if sparse_best_gain > 0 and float(best["mean_latent_variance_delta"]) > 0
        else "not supported"
        if sparse_best_gain <= 0
        else "mixed"
    )
    amp_tables = []
    for path in root.glob("*/amp_loss_tests.csv"):
        table = pd.read_csv(path)
        table["branch"] = path.parent.name
        amp_tables.append(table)
    amp_frame = pd.concat(amp_tables, ignore_index=True) if amp_tables else pd.DataFrame()
    if len(amp_frame):
        finite_mask = amp_frame["finite"].map(
            lambda value: value if isinstance(value, bool) else str(value).lower() == "true"
        )
        unstable = amp_frame.loc[
            ~finite_mask, ["branch", "dtype", "component"]
        ].to_dict("records")
    else:
        unstable = []
    tradeoffs = {
        "amplitude": float(best["mean_amplitude_delta"]),
        "correlation": float(best["mean_corr_delta"]),
        "semantic": float(best["mean_semantic_delta"]),
        "foreground_mse": float(best["mean_foreground_mse_delta"]),
    }
    report = f'''# Stage 4 sparse spatial-loss ablation report

Resolved commit: `{RESOLVED_COMMIT}`  
Fast ablation: `{FAST_ABLATION}`  
Automatic gradient balancing: `{AUTO_GRADIENT_BALANCE}`

## 1. Does dense MSE causally contribute to low-amplitude collapse?

Evidence is **{causal}** under this controlled intervention. Across {analysis_scope}, the best non-baseline spatial objective changes validation top-5 Dice by `{sparse_best_gain:+.6f}` and raw latent variance ratio by `{float(best["mean_latent_variance_delta"]):+.6f}` relative to dense MSE. This conclusion is restricted to the selected branches and fixed Stage 4 resources.

## 2. Which spatial loss most improves sparse maps?

`{best["run_name"]}` (`{best["objective"]}`) has the largest mean validation top-5 Dice improvement across PubMed and Nilearn. Foreground MSE changes by `{float(best["mean_foreground_mse_delta"]):+.6f}` (negative is better).

## 3. Dice trade-offs

For that objective, raw prediction absolute amplitude changes by `{tradeoffs["amplitude"]:+.6f}`, spatial correlation by `{tradeoffs["correlation"]:+.6f}`, and generated-map semantic normalized recall AUC by `{tradeoffs["semantic"]:+.6f}`. These jointly determine whether Dice gains reflect useful spatial recovery rather than support reshuffling.

## 4. Does NeuroVault benefit less?

Best sparse-domain Dice gain: `{sparse_best_gain:+.6f}`. Best NeuroVault Dice gain: `{nv_best_gain:+.6f}`. NeuroVault benefits less: `{sparse_best_gain > nv_best_gain}`.

## 5. BF16/FP16 stability

Non-finite component/dtype cases: `{json.dumps(unstable)}`. An empty list means all tested raw latent and spatial components had finite real-batch gradients. Training-history gradient curves should still be inspected for late-run explosions.

## Selection and interpretation safeguards

- Every reported run uses a validation-selected checkpoint; the test split was not used for selection.
- Raw decoder output is used for all training objectives. Clamping occurs only for evaluation and the released Stage 3 semantic encoder.
- Gradient-calibration coefficients are fixed before training and recorded in `gradient_calibration.json`.
- This notebook does not modify the production Stage 4 loss.
'''
    (root / "final_report.md").write_text(report.strip() + "\n")
    atomic_write_csv(root / "objective_summary.csv", by_objective.to_dict("records"))
    display(by_objective.sort_values("mean_top5_delta", ascending=False))
    print((root / "final_report.md").read_text())

build_final_outputs(EXPERIMENT_ROOT)
atomic_write_json(
    ACTIVE_POINTER,
    {"path": str(EXPERIMENT_ROOT), "state": "completed", "updated_at": utc_stamp()},
)


## Artifact checklist

The experiment root contains `effective_config.json`, `provenance.json`, `gradient_calibration.json`, `per_run_summary.csv`, `all_runs_comparison.csv`, cross-domain `plots/`, and `final_report.md`.

Every run contains:

- `effective_config.json`
- `provenance.json`
- `gradient_calibration.json`
- `training_history.csv`
- `validation_metrics.csv`
- `test_metrics.csv`
- `per_run_summary.csv`
- `checkpoint_manifest.json` and five independently managed checkpoints
- `plots/`
- `examples/`

Branch directories also retain strict pairing audits, BF16/FP16 loss tests, the ordered raw-latent cache, and 32-example tiny-control results.
